# AI Agent Tools from Pydantic Models


This tutorial demonstrates how to use the `FunctionTool.from_pydantic` method to create a function tool from a clearly-defined Pydantic class. This method is useful when you want to wrap a Python function into the `FunctionTool` object, while defining the input and output types of the function and its functionality directly. This gives you a trust way to ensure that the tools sent to `llms` are working as expected.

If you're opening this Notebook on colab, you will probably need to install `SlackAgents`. You can do this by running the following cell:

```python
!pip install slackagents
```

## Import necessary modules

In [1]:
from slackagents.tools.function_tool import FunctionTool
from pydantic import BaseModel, Field

## Define a Pydantic model for the function
In this tutorial, we'll create a function tool from a Pydantic model. Pydantic is a data validation library that uses Python type annotations for data validation and settings management. We'll define a Pydantic model for our function, specifying its input and output types. This approach offers several benefits:

1. Ensures strictly that the tool JSON file sent to LLMs matches our expectations
2. Provides better visualization of the tool definition
3. Enables automatic input validation

By using a Pydantic model, we can create a strict and type-safe function tool compared to the previous tutorial's approach of wrapping a Python function directly. Let us re-use the area calculation function from the previous tutorial `tool_from_python_function.ipynb`and define a Pydantic model for it.

```python
def calculate_area(length: float, width: float) -> float:
    """
    Calculate the area of a rectangle.
    
    :param length: The length of the rectangle.
    :param width: The width of the rectangle.
    :return: The area of the rectangle.
    """
    return length * width
```

In [5]:
class CalculateArea(BaseModel):
    length: float = Field(..., description="Length of the rectangle")
    width: float = Field(..., description="Width of the rectangle")
    
    @classmethod
    def execute(cls, length: float, width: float):
        return length * width

Note that the function arguments, their types and descriptions are defined in the `CalculateArea` Pydantic model. The `FunctionTool.from_pydantic` method is used to create a function tool from the Pydantic model. The `FunctionTool.from_pydantic` method takes the Pydantic model as an argument and returns a `FunctionTool` object.

In [6]:
area_tool = FunctionTool.from_pydantic(
    model=CalculateArea, 
    name="calculate_area", 
    description="Calculate the area of a rectangle"
)

In [7]:
import json
print(json.dumps(area_tool.info, indent=4))

{
    "type": "function",
    "function": {
        "name": "calculate_area",
        "description": "Calculate the area of a rectangle",
        "parameters": {
            "properties": {
                "length": {
                    "description": "Length of the rectangle",
                    "title": "Length",
                    "type": "number"
                },
                "width": {
                    "description": "Width of the rectangle",
                    "title": "Width",
                    "type": "number"
                }
            },
            "required": [
                "length",
                "width"
            ],
            "title": "CalculateArea",
            "type": "object",
            "additionalProperties": false
        },
        "strict": true
    }
}


To set an argument as optional, you can use the `Optional` type from the `typing` module. In the example below, the `width` argument is set as optional by using `Optional[float]`.

In [8]:
from typing import Optional
class CalculateArea(BaseModel):
    length: float = Field(..., description="Length of the rectangle")
    width: Optional[float] = Field(..., description="Width of the rectangle")
    
    @classmethod
    def execute(cls, length: float, width: float=1.0):
        return length * width

In [9]:
area_tool = FunctionTool.from_pydantic(
    model=CalculateArea, 
    name="calculate_area", 
    description="Calculate the area of a rectangle"
)

In [10]:
print(json.dumps(area_tool.info, indent=4))

{
    "type": "function",
    "function": {
        "name": "calculate_area",
        "description": "Calculate the area of a rectangle",
        "parameters": {
            "properties": {
                "length": {
                    "description": "Length of the rectangle",
                    "title": "Length",
                    "type": "number"
                },
                "width": {
                    "anyOf": [
                        {
                            "type": "number"
                        },
                        {
                            "type": "null"
                        }
                    ],
                    "description": "Width of the rectangle",
                    "title": "Width"
                }
            },
            "required": [
                "length",
                "width"
            ],
            "title": "CalculateArea",
            "type": "object",
            "additionalProperties": false
        },
        

## Execute the tool
To execute the tool, we need to create a ToolCall object and then use the execute method:

In [11]:
from slackagents.tools.base import ToolCall
call = ToolCall(name="calculate_area", arguments={"length": 5, "width": 3})

In [12]:
result = area_tool.execute(call)
print(f"The area is: {result}")

The area is: 15


## Automatic error handling
Let's see what happens when we provide invalid arguments:

In [14]:
invalid_call = ToolCall(name="calculate_area", arguments={"length": "5"})
error_result = area_tool.execute(invalid_call)
print(f"Error: {error_result}")

Error: {'error': "can't multiply sequence by non-int of type 'float'"}


# Conclusion
In this tutorial, we learned how to create a function tool from a Pydantic model. We defined a Pydantic model for the area calculation function and used the `FunctionTool.from_pydantic` method to create a function tool from the Pydantic model. We also saw how the Pydantic model helps in automatic input validation and error handling.